<a href="https://colab.research.google.com/github/imagra93/ML-course-labs/blob/main/machina_learning/lab%20-%20Polynomial%20Regression%20and%20Regularisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Polynomial Regression, Overfitting and Regularisation

This lab tells one story in three acts:

1. **Overfitting.** We fit polynomials of increasing degree to a small noisy dataset. Training error keeps falling, but test error first falls and then **explodes**.
2. **Diagnosis.** Overfitted models have **huge coefficients**. They bend wildly to pass through every noisy point.
3. **Cure.** **Regularisation** penalises large coefficients. We keep the very flexible degree-14 model, and a single knob $\lambda$ brings the test error back down. Cross-validation picks $\lambda$ without touching the test set.

### Learning objectives
- Recognise underfitting and overfitting from **train vs. test metrics**
- Understand why the gap between train and test error is what matters
- See how Ridge ($L_2$) and Lasso ($L_1$) control model complexity
- Choose the regularisation strength properly, with cross-validation

## Background

**Polynomial regression is just linear regression on new features.** For a degree-$d$ model we map each input to

$$\phi(x) = [x,\, x^2,\, \ldots,\, x^d]^\top, \qquad \hat{y} = \theta_0 + \boldsymbol{\theta}^\top \phi(x).$$

The model is non-linear in $x$ but **linear in $\boldsymbol{\theta}$**, so ordinary least squares still applies:

$$\min_{\boldsymbol{\theta}} \; \frac{1}{m}\sum_{i=1}^m \bigl(y^{(i)} - \hat{y}^{(i)}\bigr)^2 .$$

**Regularisation** adds a penalty on the size of the coefficients, weighted by $\lambda \ge 0$:

| Method | Objective | Effect |
|---|---|---|
| OLS | $\text{MSE}(\boldsymbol{\theta})$ | Fits the training data as closely as possible |
| **Ridge** ($L_2$) | $\text{MSE}(\boldsymbol{\theta}) + \lambda \lVert\boldsymbol{\theta}\rVert_2^2$ | Shrinks all coefficients smoothly toward 0 |
| **Lasso** ($L_1$) | $\text{MSE}(\boldsymbol{\theta}) + \lambda \lVert\boldsymbol{\theta}\rVert_1$ | Sets some coefficients **exactly** to 0 (feature selection) |

The intercept $\theta_0$ is not penalised. $\lambda = 0$ gives back OLS, and $\lambda \to \infty$ forces every coefficient to 0, which leaves a flat line.

> In scikit-learn the regularisation strength $\lambda$ is called `alpha`.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import LinAlgWarning

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

# Tiny-lambda Ridge and Lasso emit numerical warnings that are expected here
warnings.filterwarnings('ignore', category=LinAlgWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

plt.style.use('seaborn-v0_8-whitegrid')

### Helpers

Every model in this lab is the same three-step pipeline:

1. `PolynomialFeatures`: $x \mapsto [x, x^2, \ldots, x^d]$
2. `StandardScaler`: puts every feature on the same scale. On $[0, 1]$, $x^{14}$ is tiny compared with $x$, so without scaling the penalty would hit some features far harder than others.
3. A linear estimator: `LinearRegression`, `Ridge` or `Lasso`

Scaling does not change what plain OLS predicts. It matters only once we add a penalty.

In [ ]:
def make_model(degree, estimator=None):
    """Polynomial features -> standardisation -> linear estimator (OLS by default)."""
    return make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        StandardScaler(),
        estimator if estimator is not None else LinearRegression(),
    )

def train_test_mse(model):
    """Fit on the training set, return (train MSE, test MSE)."""
    model.fit(x_train, y_train)
    return (mean_squared_error(y_train, model.predict(x_train)),
            mean_squared_error(y_test,  model.predict(x_test)))

def plot_fit(ax, model, title):
    """Plot the fitted curve (on a dense grid) against the data and the true function."""
    x_grid = np.linspace(0, 1, 500).reshape(-1, 1)
    ax.plot(x_grid, true_f(x_grid), 'k--', lw=1, label='True function')
    ax.scatter(x_test,  y_test,  s=4,  c='C3', alpha=0.15, label='Test')
    ax.scatter(x_train, y_train, s=40, c='C0', edgecolor='k', zorder=3, label='Train')
    ax.plot(x_grid, model.predict(x_grid), 'C2', lw=2.5, label='Model')
    ax.set_ylim(-2, 2)
    ax.set_xlabel('x')
    ax.set_title(title, fontsize=11)

## The data

We use the classic setup from Bishop's *Pattern Recognition and Machine Learning*:

$$y = \sin(2\pi x) + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0,\, 0.3^2)$$

- **Training set:** only **15 points**, evenly spaced on $[0, 1]$. Small datasets are where overfitting bites.
- **Test set:** 1000 fresh points from the same distribution. A test set this large is a luxury of synthetic data, but it makes test MSE a reliable estimate of the true generalisation error.

The noise sets a floor. Even the true function has an expected MSE of $\sigma^2 = 0.09$ on new data, so **no model can beat a test MSE of about 0.09**.

In [ ]:
rng = np.random.default_rng(0)

def true_f(x):
    return np.sin(2 * np.pi * x)

NOISE_STD = 0.3
N_TRAIN   = 15

x_train = np.linspace(0, 1, N_TRAIN).reshape(-1, 1)
y_train = true_f(x_train).ravel() + rng.normal(0, NOISE_STD, N_TRAIN)

x_test = rng.uniform(0, 1, (1000, 1))
y_test = true_f(x_test).ravel() + rng.normal(0, NOISE_STD, 1000)

NOISE_FLOOR = NOISE_STD ** 2   # best achievable test MSE

fig, ax = plt.subplots(figsize=(9, 4))
x_grid = np.linspace(0, 1, 500)
ax.plot(x_grid, true_f(x_grid), 'k--', lw=1, label='True function')
ax.scatter(x_test,  y_test,  s=4,  c='C3', alpha=0.2, label='Test (1000 points)')
ax.scatter(x_train, y_train, s=40, c='C0', edgecolor='k', zorder=3, label='Train (15 points)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend()
plt.show()

---
# Act 1: Overfitting

## Underfit, good fit, overfit

We fit three OLS models of degree 1, 3 and 14 and look at **both** metrics:

- **Train MSE:** how well the model fits the data it has seen
- **Test MSE:** how well it predicts data it has **never seen**. This is the number we actually care about.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, degree, label in zip(axes, [1, 3, 14], ['Underfitting', 'Good fit', 'Overfitting']):
    model = make_model(degree)
    tr, te = train_test_mse(model)
    plot_fit(ax, model, f'{label}: degree {degree}\ntrain MSE = {tr:.3f}   test MSE = {te:.3f}')

axes[0].legend(loc='lower left')
plt.tight_layout()
plt.show()

| Degree | What the model does | Train MSE | Test MSE | Diagnosis |
|---|---|---|---|---|
| 1 | A straight line cannot follow a sine wave | high | high | **Underfitting** (high bias) |
| 3 | Follows the shape and ignores the noise | low | low (close to the 0.09 floor) | **Good fit** |
| 14 | 15 parameters for 15 points: it passes through **every** training point, noise included | **≈ 0** | **very high** | **Overfitting** (high variance) |

**Key insight:** training error alone is misleading. The degree-14 model has the *best* training error and the *worst* test error. It has **memorised** the noise instead of learning the pattern.

## Train vs. test error as model complexity grows

Now we sweep every degree from 1 to 14. We also record the size of the coefficients, $\lVert\boldsymbol{\theta}\rVert_2$, which becomes important in Act 2.

In [ ]:
degrees = np.arange(1, 15)
train_mse, test_mse, coef_norm = [], [], []

for d in degrees:
    model = make_model(d)
    tr, te = train_test_mse(model)
    train_mse.append(tr)
    test_mse.append(te)
    coef_norm.append(np.linalg.norm(model[-1].coef_))

best_degree = degrees[np.argmin(test_mse)]

print(' Degree |  Train MSE |   Test MSE |  Gap (test - train) |  ||theta||')
print('--------+------------+------------+---------------------+-----------')
for d, tr, te, cn in zip(degrees, train_mse, test_mse, coef_norm):
    print(f'   {d:>2}   |   {tr:.4f}   | {te:>10.4f} | {te - tr:>19.4f} | {cn:>9.1e}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))

ax1.plot(degrees, train_mse, 'o-', c='C0', lw=2, label='Train MSE')
ax1.plot(degrees, test_mse,  's-', c='C3', lw=2, label='Test MSE')
ax1.axhline(NOISE_FLOOR, color='gray', ls=':', label=r'Noise floor $\sigma^2$')
ax1.fill_between(degrees, train_mse, test_mse, color='C3', alpha=0.08, label='Generalisation gap')
ax1.axvline(best_degree, color='C2', ls='--', label=f'Best test MSE (degree {best_degree})')
ax1.set_yscale('log')
ax1.set_ylim(1e-3, 5)   # degree-14 train MSE is ~1e-13 and falls off the bottom
ax1.set_xlabel('Polynomial degree  (model complexity →)')
ax1.set_ylabel('MSE (log scale)')
ax1.set_title('Train error always falls; test error is U-shaped')
ax1.text(1.2, 2, 'UNDERFIT', color='gray', fontsize=11)
ax1.text(11.5, 2, 'OVERFIT', color='gray', fontsize=11)
ax1.legend(loc='lower left', fontsize=9)

ax2.semilogy(degrees, coef_norm, 'D-', c='C4', lw=2)
ax2.set_xlabel('Polynomial degree')
ax2.set_ylabel(r'$\|\theta\|_2$ (log scale)')
ax2.set_title('Overfitted models have enormous coefficients')

plt.tight_layout()
plt.show()

### Reading the plots

- **Training MSE decreases monotonically.** A more flexible model can always fit the training data at least as well. At degree 14 it reaches zero.
- **Test MSE is U-shaped.** It first drops as the model becomes flexible enough to follow the sine (less **bias**), then shoots up as the model starts fitting noise (more **variance**).
- **The generalisation gap** (test − train) is the signature of overfitting: a small gap with high error means underfitting, a large gap means overfitting.
- **The coefficients explode**, from about $10^0$ at degree 1 to about $10^9$ at degree 14. To wiggle through every noisy point, the model needs huge positive and negative terms that nearly cancel. Between the training points those terms stop cancelling, and the curve swings wildly.

This last observation suggests a cure. **If overfitting shows up as huge coefficients, penalise huge coefficients.**

---
# Act 2: Regularisation

We could simply pick degree 3. In real problems, though, we rarely know the "right" complexity. A popular alternative is to **keep a flexible model and restrain it**.

From here on, the model is always the **degree-14 polynomial** that overfitted so badly. The only thing we change is the Ridge penalty $\lambda$:

$$\min_{\boldsymbol{\theta}} \; \text{MSE}(\boldsymbol{\theta}) + \lambda \lVert\boldsymbol{\theta}\rVert_2^2$$

- $\lambda = 0$: plain OLS, which overfits
- moderate $\lambda$: coefficients are kept small, so the curve is smooth
- huge $\lambda$: coefficients are crushed to 0, so the model underfits

In [ ]:
DEGREE = 14

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, lam, label in zip(axes, [0, 1e-3, 10], ['No penalty (OLS)', 'Moderate penalty', 'Strong penalty']):
    model = make_model(DEGREE, Ridge(alpha=lam) if lam > 0 else None)
    tr, te = train_test_mse(model)
    plot_fit(ax, model, f'{label}: $\\lambda$ = {lam:g}\ntrain MSE = {tr:.3f}   test MSE = {te:.3f}')

axes[0].legend(loc='lower left')
plt.suptitle(f'Same degree-{DEGREE} model, different Ridge penalties', y=1.03)
plt.tight_layout()
plt.show()

The same 15 features, with a penalty added, take test MSE from about 2.4 down to about 0.13. We did not remove any feature. We only stopped the coefficients from growing huge.

## Train vs. test error as $\lambda$ grows

We sweep $\lambda$ over 16 orders of magnitude. This plot mirrors the degree plot from Act 1, except that **complexity now decreases from left to right**.

In [ ]:
lambdas = np.logspace(-14, 2, 50)
ridge_train, ridge_test, ridge_norm = [], [], []

for lam in lambdas:
    model = make_model(DEGREE, Ridge(alpha=lam))
    tr, te = train_test_mse(model)
    ridge_train.append(tr)
    ridge_test.append(te)
    ridge_norm.append(np.linalg.norm(model[-1].coef_))

ols_train, ols_test = train_test_mse(make_model(DEGREE))
best_lambda_test = lambdas[np.argmin(ridge_test)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))

ax1.semilogx(lambdas, ridge_train, c='C0', lw=2, label='Train MSE')
ax1.semilogx(lambdas, ridge_test,  c='C3', lw=2, label='Test MSE')
ax1.axhline(ols_test, color='C3', ls='-.', alpha=0.6, label=f'Test MSE without penalty = {ols_test:.2f}')
ax1.axhline(NOISE_FLOOR, color='gray', ls=':', label=r'Noise floor $\sigma^2$')
ax1.fill_between(lambdas, ridge_train, ridge_test, color='C3', alpha=0.08)
ax1.set_yscale('log')
ax1.set_ylim(1e-4, 5)
ax1.set_xlabel(r'$\lambda$  (← more complex      simpler →)')
ax1.set_ylabel('MSE (log scale)')
ax1.set_title(f'Ridge on degree-{DEGREE} features')
ax1.text(1e-13, 1.2, 'OVERFIT', color='gray', fontsize=11)
ax1.text(3e-1, 1.2, 'UNDERFIT', color='gray', fontsize=11)
ax1.legend(loc='lower right', fontsize=9)

ax2.loglog(lambdas, ridge_norm, c='C4', lw=2)
ax2.set_xlabel(r'$\lambda$')
ax2.set_ylabel(r'$\|\theta\|_2$ (log scale)')
ax2.set_title('Larger penalty, smaller coefficients')

plt.tight_layout()
plt.show()

### Reading the plots

- **Every Ridge model beats OLS by a wide margin.** Even $\lambda = 10^{-14}$ is far below the "no penalty" line, because a tiny penalty is already enough to stop the coefficients from reaching $10^{9}$.
- **The same U-shape appears**, flipped horizontally. Small $\lambda$ gives low train error and a large gap (overfit). Large $\lambda$ makes train and test error both high (underfit). In between, test MSE gets close to the noise floor.
- **$\lambda$ is a complexity dial.** Degree is a coarse, discrete dial. $\lambda$ is a continuous one, and it lets us keep a rich feature set.

But notice what we just did: we picked the best $\lambda$ **by looking at the test set**. That is cheating. Once the test set is used to choose the model, it no longer gives an unbiased estimate of performance on new data.

## Choosing $\lambda$ properly: cross-validation

We pick $\lambda$ using **only the training data**, with $k$-fold cross-validation:

1. Split the 15 training points into $k = 5$ folds.
2. For each candidate $\lambda$: train on 4 folds, measure MSE on the held-out fold, rotate, and average.
3. Pick the $\lambda$ with the lowest average validation MSE, then refit on all training data.
4. Evaluate on the test set **once**, at the very end.

`GridSearchCV` does all of this. The scaler sits inside the pipeline, so it is also re-fit on each fold and nothing leaks from the validation fold.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
search = GridSearchCV(
    make_model(DEGREE, Ridge()),
    param_grid={'ridge__alpha': np.logspace(-10, 2, 49)},
    cv=cv,
    scoring='neg_mean_squared_error',
)
search.fit(x_train, y_train)

best_lambda = search.best_params_['ridge__alpha']
ridge_cv_model = search.best_estimator_
print(f'lambda chosen by 5-fold CV : {best_lambda:.1e}')
print(f'lambda best on test (cheat): {best_lambda_test:.1e}')

## Final comparison

All models are scored on the same test set. None of them used it for model selection.

In [ ]:
candidates = {
    'OLS, degree 1  (underfit)':          make_model(1),
    'OLS, degree 3  (hand-picked)':       make_model(3),
    f'OLS, degree {DEGREE} (overfit)':    make_model(DEGREE),
    f'Ridge, degree {DEGREE}, CV lambda': make_model(DEGREE, Ridge(alpha=best_lambda)),
}

print(f'{"Model":<34} | Train MSE | Test MSE |   Gap')
print('-' * 34 + '-+-----------+----------+--------')
for name, model in candidates.items():
    tr, te = train_test_mse(model)
    print(f'{name:<34} |  {tr:.4f}   | {te:>8.4f} | {te - tr:>6.3f}')
print(f'\n(Noise floor = {NOISE_FLOOR:.2f}: no model can do better on average.)')

The regularised degree-14 model **matches the hand-picked degree-3 model** without us having to know that the answer was "about degree 3". Cross-validation found the right amount of restraint from the training data alone.

## Ridge vs. Lasso: where do the coefficients go?

Both penalties fight overfitting, but they shrink coefficients differently:

- **Ridge ($L_2$)** shrinks every coefficient a little. None become exactly zero.
- **Lasso ($L_1$)** pushes many coefficients to **exactly zero**, so it effectively chooses which powers of $x$ to use.

Geometrically, the $L_1$ constraint region $\lVert\boldsymbol{\theta}\rVert_1 \le t$ is a diamond with corners on the axes. The loss contours usually touch it at a corner, where some $\theta_j = 0$. The $L_2$ region is a sphere with no corners.

In [ ]:
lasso_search = GridSearchCV(
    make_model(DEGREE, Lasso(max_iter=200_000)),
    param_grid={'lasso__alpha': np.logspace(-5, 0, 26)},
    cv=cv,
    scoring='neg_mean_squared_error',
).fit(x_train, y_train)
lasso_model = lasso_search.best_estimator_

ridge_coef = ridge_cv_model[-1].coef_
lasso_coef = lasso_model[-1].coef_
powers = np.arange(1, DEGREE + 1)

_, ridge_te = train_test_mse(make_model(DEGREE, Ridge(alpha=best_lambda)))
_, lasso_te = train_test_mse(make_model(DEGREE, Lasso(alpha=lasso_search.best_params_['lasso__alpha'], max_iter=200_000)))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].bar(powers, ridge_coef, color='C0', edgecolor='k')
axes[0].set_title(f'Ridge ($\\lambda$={best_lambda:.1e}): {np.sum(ridge_coef != 0)}/{DEGREE} non-zero, '
                  f'test MSE = {ridge_te:.3f}')
axes[1].bar(powers, lasso_coef, color='C1', edgecolor='k')
axes[1].set_title(f'Lasso ($\\lambda$={lasso_search.best_params_["lasso__alpha"]:.1e}): '
                  f'{np.sum(lasso_coef != 0)}/{DEGREE} non-zero, test MSE = {lasso_te:.3f}')
for ax in axes:
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(powers)
    ax.set_xticklabels([f'$x^{{{p}}}$' for p in powers])
    ax.set_ylabel('Coefficient (standardised features)')
plt.tight_layout()
plt.show()

Both reach a similar test error. Ridge spreads the weight across all 14 powers, while Lasso uses only a handful and sets the rest to zero. When you suspect that only a few features really matter, and you want an interpretable model, Lasso is the natural choice.

---
## Summary

| Symptom | Train MSE | Test MSE | Gap | Remedy |
|---|---|---|---|---|
| **Underfitting** | high | high | small | More flexible model, smaller $\lambda$ |
| **Good fit** | low | low (near noise floor) | small | Keep it |
| **Overfitting** | very low | high | **large** | Simpler model, **larger $\lambda$**, more data |

1. **Always report train *and* test metrics.** Low training error on its own proves nothing.
2. **Overfitting = large generalisation gap.** In linear models it comes with exploding coefficients.
3. **Regularisation penalises coefficient size**, which gives a continuous complexity dial $\lambda$.
4. **Choose $\lambda$ (or the degree) with cross-validation** on the training data, never on the test set.

## Exercises

1. **More data cures overfitting.** Change `N_TRAIN` from 15 to 200 and re-run the notebook. What happens to the degree-14 OLS test error, and to the gap? Why?
2. **More noise.** Set `NOISE_STD = 0.6`. Does the best degree go up or down? Does the CV-chosen $\lambda$ go up or down?
3. **Forget the scaler.** Remove `StandardScaler()` from `make_model` and re-run the Ridge sweep. Why does the behaviour change so much? (Hint: compare the scale of $x$ and $x^{14}$ on $[0, 1]$.)
4. **Bias-variance by simulation.** Draw 100 different training sets and fit a degree-1, a degree-3 and a degree-14 model on each. Plot all 100 curves per degree. Which degree has curves that disagree the most with each other (variance)? Which is consistently wrong (bias)?
5. **Ridge and conditioning.** The Ridge solution is $\boldsymbol{\theta}^* = (\boldsymbol{\Phi}^\top\boldsymbol{\Phi} + \lambda\mathbf{I})^{-1}\boldsymbol{\Phi}^\top\mathbf{y}$. Why is this matrix always invertible for $\lambda > 0$, even when $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$ is not?

## Answers

**1.** With 200 points, 15 parameters can no longer pass through every point, so the model has to average out the noise. The degree-14 test MSE drops close to the noise floor and the gap nearly vanishes. Overfitting is a problem of *too many parameters for the amount of data*, not of high degree as such.

**2.** More noise makes the data less informative, so the optimal model gets *simpler*: a lower best degree and a **larger** CV-chosen $\lambda$.

**3.** Without scaling, the high powers of $x$ are tiny (for example $0.5^{14} \approx 6 \times 10^{-5}$). The model then needs enormous coefficients on them to have any effect, and the penalty treats features unevenly. The same $\lambda$ means very different things for different features, and good values of $\lambda$ shift by orders of magnitude.

**4.** Degree 14 curves vary wildly between datasets: **high variance**. Degree 1 curves all look alike but are all wrong: **high bias**. Degree 3 has both low. Expected test error = bias² + variance + $\sigma^2$.

**5.** $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$ is positive semi-definite, with eigenvalues $\mu_i \ge 0$. Adding $\lambda\mathbf{I}$ shifts every eigenvalue to $\mu_i + \lambda \ge \lambda > 0$, so the matrix is positive definite and therefore invertible. It is also better conditioned: $\kappa = \frac{\mu_{\max} + \lambda}{\mu_{\min} + \lambda}$, which is much smaller than $\mu_{\max}/\mu_{\min}$.